# マーク付き点過程モデル with 距離事前分布

このノートブックでは、距離事前分布を組み込んだマーク付き点過程モデルを実データ（黒曜石）で実行します。

## モデル概要

### 点過程部分（Intensity Process）
- 遺跡の存在確率：$q(s) = \text{sigmoid}(\eta_{\text{int}})$
- 強度関数：$\lambda(s) = \lambda^* \cdot q(s)$
- 偽不在点のサンプリング：$U \sim \text{IPP}(\lambda^*(1-q))$

### マーク部分（Composition Process）
- 産地構成比：$\pi_k(s) = \text{softmax}(\eta_k)$
- **距離事前分布**：切片に産地からの距離情報を組み込む
  - $\beta_{0k}(s) \sim \text{GP}(\lambda_k \cdot g_k(s), C)$
  - $g_k(s)$：距離ベースのlog-ratio特徴量
  - データ豊富な領域：事後分布がデータに引っ張られる
  - データ希薄な領域：距離事前分布に近い値を保つ

### 空間効果
- NNGP（Nearest Neighbor Gaussian Process）による空間相関
- 各特徴量ごとに異なるカーネルパラメータを設定可能

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import japanize_matplotlib

from bayesian_statistics.models.preprocessing.data_preprocessor import (
    ObsidianDataPreprocessor,
)
from bayesian_statistics.nngp.model.marked_point_process import (
    MarkedPointProcessConfig,
    MarkedPointProcessSampler,
    compute_eta,
    prepare_marked_point_process_dataset,
    softmax_with_baseline,
)

print("モジュール読み込み完了")

## 1. データの準備

In [ ]:
data_dir = "../data"
preprocessor = ObsidianDataPreprocessor(data_dir, scale_variables=True)
preprocessor.load_data()

period = 2  # 中期
origins = ["神津島", "信州", "箱根", "高原山", "その他"]
time_periods = {0: "早期・早々期", 1: "前期", 2: "中期", 3: "後期", 4: "晩期"}

print(f"時期: {time_periods[period]}")
print(f"産地: {origins}")

## 2. ハイパーパラメータの設定

### カーネルパラメータの意味

- **lengthscale（レンジ）**: 波のゆったりさ、空間的な影響範囲
- **variance（分散）**: 波の高さ、効果の大きさ

| lengthscale | variance | 効果 |
|:-----------|:--------|:-----|
| 小 | 大 | 強い局所効果：局所的なデータに強く反応し、大きく変動 |
| 小 | 小 | 弱い局所効果：局所的だが変動は小さい |
| 大 | 大 | 強い広域効果：広域的に滑らかで、大きく変動 |
| 大 | 小 | 弱い広域効果：広域的に滑らかで、変動は小さい |

In [ ]:
# ========== ハイパーパラメータ設定 ==========

# グリッド点のサブサンプリング比率
grid_subsample_ratio = 0.01  # 1%をサンプリング（計算時間短縮）

# 距離列名（4つの主要産地のみ）
distance_column_names = [
    "cost_kouzu",
    "cost_shinshu",
    "cost_hakone",
    "cost_takahara",
]

# 距離事前分布のハイパーパラメータ
tau = 0.5  # 温度パラメータ（小さいほど距離による差が大きい）
alpha = 1.0  # 重要度の指数
source_weights = [2, 1, 0.05, 0.05]  # 産地の重要度（神津島＞信州＞箱根≒高原山）
lambda_fixed = [1, 1, 1, 1]  # λの固定値（K-1=4）

# 強度モデル（点過程）の共変量
intensity_variable_names = ["average_elevation", "average_slope_angle"]

# マークモデル（組成）の共変量（距離特徴量のみ使用）
mark_variable_names = None  # 距離事前分布のみ

# カーネルパラメータ（マーク用）
# intercept: 距離事前分布＋データ駆動調整
mark_lengthscale = 0.2  # ベースライン（チューニング対象）
mark_variance = 0.1  # ベースライン（チューニング対象）

# カーネルパラメータ（強度用）
intensity_lengthscale = 0.1
intensity_variance = 1.0

# MCMC設定
n_iter = 1000
burn_in = 200
thinning = 2
neighbor_count = 25

# λ*の事前分布（Gamma(shape, rate)）
lambda_prior_shape = 2.0
lambda_prior_rate = 0.1

print("ハイパーパラメータ設定完了")
print(f"  保存サンプル数: {(n_iter - burn_in) // thinning}")
print(f"  距離事前分布: tau={tau}, alpha={alpha}")
print(f"  産地重要度: {source_weights}")

## 3. データセットの準備

In [ ]:
dataset = prepare_marked_point_process_dataset(
    preprocessor=preprocessor,
    period=period,
    origins=origins,
    grid_subsample_ratio=grid_subsample_ratio,
    drop_zero_total_sites=True,
    intensity_variable_names=intensity_variable_names,
    mark_variable_names=mark_variable_names,
    distance_column_names=distance_column_names,
    source_weights=source_weights,
    lambda_fixed=lambda_fixed,
    tau=tau,
    alpha=alpha,
)

print(f"遺跡数: {dataset.num_sites()}")
print(f"カテゴリ数（K）: {dataset.num_categories()}")
print(f"グリッド数: {dataset.num_grid()}")
print(f"有効グリッド数: {dataset.valid_grids.sum()}")
print(f"領域面積: {dataset.volume:.4f}")
print(f"\n距離特徴量の形状: {dataset.distance_features_sites.shape}")
print(f"事前平均の形状: {dataset.prior_mean_intercept_sites.shape}")

In [ ]:
# 産地別出土数
counts_sum = dataset.counts.sum(axis=0)
print("産地別出土数:")
for k, origin in enumerate(origins):
    pct = counts_sum[k] / counts_sum.sum() * 100
    print(f"  {origin}: {int(counts_sum[k])} ({pct:.1f}%)")

## 4. 境界データと陸地マスクの準備

In [ ]:
# 境界（可視化用）
boundary = (
    preprocessor.df_elevation.filter(
        pl.col("average_elevation").is_null(), ~pl.col("is_sea")
    )
    .select(["x", "y"])
    .to_numpy()
)

# 陸地マスク
is_land = dataset.valid_grids

print(f"境界点数: {len(boundary)}")
print(f"陸地グリッド数: {is_land.sum()}")

## 5. 距離ベース事前確率の可視化

モデルが使用する距離ベース基準確率 $p_{0k}$ を確認します。
これはデータがない場合の「ベースライン期待値」です。

In [ ]:
from bayesian_statistics.nngp.model.sample import logratio_to_probs

# グリッド上の距離ベース基準確率 p_0
# distance_features_gridは既にlog-ratio形式 g_k = log(p_0k) - log(p_0K)
# 元の基準確率 p_0 を復元するには、lambda=1として単純に確率に戻す
p0_grid = logratio_to_probs(dataset.distance_features_grid.T)
p0_grid = p0_grid.T  # (n_grid, K)

def plot_distance_prior(origin_index: int):
    fig, ax = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)

    ax.scatter(
        dataset.grid_coords[is_land, 0],
        dataset.grid_coords[is_land, 1],
        c=p0_grid[is_land, origin_index],
        cmap="Reds",
        s=10,
        alpha=0.8,
        vmin=0,
        vmax=1,
    )
    ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)

    plt.colorbar(ax.collections[0], ax=ax, label="距離ベース事前確率")
    ax.set_title(f"距離事前分布: {origins[origin_index]}")
    ax.set_xlabel("経度")
    ax.set_ylabel("緯度")

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    plt.show()

print("距離ベース基準確率 p_0（重み付き逆ソフトマックス）:")
print(f"tau={tau}, alpha={alpha}")
print(f"産地重要度: {source_weights}")

In [ ]:
# 4つの主要産地の距離事前分布を可視化
for k in range(4):
    plot_distance_prior(k)

## 6. MCMC設定と実行

In [ ]:
config = MarkedPointProcessConfig(
    n_iter=n_iter,
    burn_in=burn_in,
    thinning=thinning,
    seed=42,
    neighbor_count=neighbor_count,
    intensity_kernel_lengthscale=intensity_lengthscale,
    intensity_kernel_variance=intensity_variance,
    mark_kernel_lengthscale=mark_lengthscale,
    mark_kernel_variance=mark_variance,
    lambda_prior_shape=lambda_prior_shape,
    lambda_prior_rate=lambda_prior_rate,
    tau=tau,
    alpha=alpha,
    source_weights=source_weights,
    lambda_fixed=lambda_fixed,
)

print(f"保存サンプル数: {config.n_saved()}")

In [ ]:
# サンプラー初期化と実行
sampler = MarkedPointProcessSampler(dataset, config)

print("MCMC実行中...")
results = sampler.run(show_progress=True)
print("完了!")
print(f"保存サンプル数: {len(results.lambda_star_samples)}")

## 7. 結果の確認

In [ ]:
# λ*のトレースプロット
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(results.lambda_star_samples)
ax.set_xlabel('サンプル番号')
ax.set_ylabel('λ*')
ax.set_title('λ* のトレースプロット')

ax = axes[1]
ax.hist(results.lambda_star_samples, bins=30, density=True, alpha=0.7)
ax.axvline(results.lambda_star_samples.mean(), color='red', linestyle='--', 
           label=f'平均: {results.lambda_star_samples.mean():.2f}')
ax.set_xlabel('λ*')
ax.set_ylabel('密度')
ax.set_title('λ* の事後分布')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# λ*のサマリー
print("=" * 50)
print("λ* の事後サマリー")
print("=" * 50)
print(f"平均:     {results.lambda_star_samples.mean():.2f}")
print(f"中央値:   {np.median(results.lambda_star_samples):.2f}")
print(f"標準偏差: {results.lambda_star_samples.std():.2f}")
print(f"95%CI:    [{np.percentile(results.lambda_star_samples, 2.5):.2f}, "
      f"{np.percentile(results.lambda_star_samples, 97.5):.2f}]")

In [ ]:
# 偽不在点の統計
print("=" * 50)
print("偽不在点の統計")
print("=" * 50)
n_U_array = np.array(results.n_pseudo_absence)
print(f"平均: {n_U_array.mean():.1f}")
print(f"最小: {n_U_array.min()}")
print(f"最大: {n_U_array.max()}")

## 8. 事後確率の計算

In [ ]:
# 事後平均で構成比を計算
site_probs = results.predict_probabilities(location="sites")
grid_probs = results.predict_probabilities(location="grid", sample_conditional=False)

# 実測比率
true_ratio = dataset.counts / dataset.counts.sum(axis=1, keepdims=True)
true_ratio = np.nan_to_num(true_ratio).T  # (K, n_sites)

print(f"推定確率 π（観測点） の形状: {site_probs.shape}")
print(f"推定確率 π（グリッド） の形状: {grid_probs.shape}")

## 9. 効果の分解

距離事前分布モデルの利点：効果を分解して解釈できる
- **distance**: 純粋な距離効果（データの影響なし）
- **intercept_adjustment**: データによる距離ベースラインからの調整
- **intercept**: 距離＋調整の合計効果
- **full**: 全効果（切片＋共変量）

In [ ]:
# 効果分解を実行
effects_grid = results.decompose_effects(location="grid")

print("利用可能な効果:")
for key in effects_grid.keys():
    print(f"  - {key}")

## 10. 事後平均の可視化

In [ ]:
def plot_grid_result(origin_index: int):
    """グリッド上の事後平均をプロット"""
    fig, ax = plt.subplots(1, 1, figsize=(10, 8), constrained_layout=True)

    # グリッド上の事後平均（陸地のみ）
    sc = ax.scatter(
        dataset.grid_coords[is_land, 0],
        dataset.grid_coords[is_land, 1],
        c=grid_probs[origin_index, is_land],
        cmap="Blues",
        s=10,
        alpha=0.8,
        vmin=0,
        vmax=1,
    )

    # 観測地点の実測比率
    ax.scatter(
        dataset.site_coords[:, 0],
        dataset.site_coords[:, 1],
        c=true_ratio[origin_index],
        cmap="Blues",
        s=40,
        edgecolors="black",
        linewidths=0.5,
        alpha=0.9,
        vmin=0,
        vmax=1,
    )
    
    # 境界
    ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)

    plt.colorbar(sc, ax=ax, label="事後平均確率")
    ax.set_title(f"{origins[origin_index]}産黒曜石の組成比（{time_periods[period]}）")
    ax.set_xlabel("経度")
    ax.set_ylabel("緯度")

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    plt.show()

# 4つの産地について可視化
for k in range(4):
    plot_grid_result(k)

## 11. 効果の比較プロット

In [ ]:
def plot_effect_comparison(origin_index: int, effect_names: list):
    """複数の効果を比較プロット"""
    n_effects = len(effect_names)
    fig, axes = plt.subplots(
        1, n_effects, figsize=(6 * n_effects, 5), constrained_layout=True
    )
    if n_effects == 1:
        axes = [axes]

    for ax, effect_name in zip(axes, effect_names):
        effect_data = effects_grid[effect_name]

        ax.scatter(
            dataset.grid_coords[is_land, 0],
            dataset.grid_coords[is_land, 1],
            c=effect_data[origin_index, is_land],
            cmap="Blues",
            s=10,
            alpha=0.8,
            vmin=0,
            vmax=1,
        )
        ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)
        plt.colorbar(ax.collections[0], ax=ax, label="確率")
        ax.set_title(f"{origins[origin_index]}: {effect_name}")
        ax.set_xlabel("経度")
        ax.set_ylabel("緯度")

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    plt.show()

# 神津島について、距離効果・調整・完全モデルを比較
plot_effect_comparison(0, ["distance", "intercept_adjustment", "full"])

In [ ]:
# 信州について
plot_effect_comparison(1, ["distance", "intercept_adjustment", "full"])

In [ ]:
# 箱根について
plot_effect_comparison(2, ["distance", "intercept_adjustment", "full"])

In [ ]:
# 高原山について
plot_effect_comparison(3, ["distance", "intercept_adjustment", "full"])

## 12. すべての産地の効果比較

In [ ]:
def plot_all_origins_single_effect(effect_name: str):
    """全産地について1つの効果を比較プロット"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()

    effect_data = effects_grid[effect_name]

    for idx in range(4):  # 4つの主要産地
        ax = axes[idx]
        ax.scatter(
            dataset.grid_coords[is_land, 0],
            dataset.grid_coords[is_land, 1],
            c=effect_data[idx, is_land],
            cmap="Blues",
            s=10,
            alpha=0.8,
            vmin=0,
            vmax=1,
        )
        ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)
        plt.colorbar(ax.collections[0], ax=ax, label="確率")
        ax.set_title(f"{origins[idx]}")
        ax.set_xlabel("経度")
        ax.set_ylabel("緯度")

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    fig.suptitle(f"効果: {effect_name}", fontsize=14)
    plt.show()

# 距離効果のみを4産地で比較
plot_all_origins_single_effect("distance")

In [ ]:
# データ駆動の調整を4産地で比較
plot_all_origins_single_effect("intercept_adjustment")

In [ ]:
# 完全モデルを4産地で比較
plot_all_origins_single_effect("full")

## 13. 推定値と実測値の比較

In [ ]:
def plot_comparison(origin_index: int):
    """推定値と実測値を並べてプロット"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    # 推定値
    ax = axes[0]
    sc = ax.scatter(
        dataset.grid_coords[is_land, 0],
        dataset.grid_coords[is_land, 1],
        c=grid_probs[origin_index, is_land],
        cmap="Blues",
        s=10,
        alpha=0.8,
        vmin=0,
        vmax=1,
    )
    ax.scatter(
        dataset.site_coords[:, 0],
        dataset.site_coords[:, 1],
        c=site_probs[origin_index],
        cmap="Blues",
        s=40,
        edgecolors="black",
        linewidths=0.5,
        alpha=0.8,
        vmin=0,
        vmax=1,
    )
    ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)
    plt.colorbar(sc, ax=ax, label="確率")
    ax.set_title(f"{origins[origin_index]}: 推定値（事後平均）")
    ax.set_xlabel("経度")
    ax.set_ylabel("緯度")
    
    # 実測値
    ax = axes[1]
    sc = ax.scatter(
        dataset.site_coords[:, 0],
        dataset.site_coords[:, 1],
        c=true_ratio[origin_index],
        cmap="Blues",
        s=40,
        alpha=0.8,
        vmin=0,
        vmax=1,
        edgecolors="black",
        linewidths=0.5,
    )
    ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)
    plt.colorbar(sc, ax=ax, label="比率")
    ax.set_title(f"{origins[origin_index]}: 実測値")
    ax.set_xlabel("経度")
    ax.set_ylabel("緯度")

    for ax in axes:
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    plt.show()

# 4産地について比較
for k in range(4):
    plot_comparison(k)

## 14. 散布図による精度評価

In [ ]:
# 真の値と推定値の比較（散布図）
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for k in range(4):
    ax = axes[k]
    ax.scatter(true_ratio[k], site_probs[k], alpha=0.6, s=30)
    ax.plot([0, 1], [0, 1], 'r--', label='y=x')
    ax.set_xlabel('実測比率')
    ax.set_ylabel('推定確率')
    ax.set_title(origins[k])
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    
    # 相関係数とRMSE
    corr = np.corrcoef(true_ratio[k], site_probs[k])[0, 1]
    rmse = np.sqrt(np.mean((site_probs[k] - true_ratio[k])**2))
    ax.text(0.05, 0.9, f'r = {corr:.3f}\nRMSE = {rmse:.3f}', 
            fontsize=10, transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# 推定精度の評価
print("=" * 50)
print("推定精度の評価")
print("=" * 50)
print()

for k, origin in enumerate(origins[:-1]):  # 「その他」を除く
    rmse = np.sqrt(np.mean((site_probs[k] - true_ratio[k])**2))
    mae = np.mean(np.abs(site_probs[k] - true_ratio[k]))
    corr = np.corrcoef(site_probs[k], true_ratio[k])[0, 1]
    
    print(f"{origin}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  相関: {corr:.4f}")
    print()

## 15. 遺跡存在強度の予測と可視化

In [ ]:
# グリッド上での強度予測
grid_intensity = results.predict_intensity(location="grid")

# 陸地のみでの統計（NaNを含む無効なグリッドを除外）
intensity_land = grid_intensity[is_land]
print(f"強度の範囲: [{intensity_land.min():.4f}, {intensity_land.max():.4f}]")
print(f"平均強度: {intensity_land.mean():.4f}")

In [ ]:
# 強度のプロット
fig, ax = plt.subplots(1, 1, figsize=(10, 8), constrained_layout=True)

# グリッド上の強度（陸地のみ）
sc = ax.scatter(
    dataset.grid_coords[is_land, 0],
    dataset.grid_coords[is_land, 1],
    c=grid_intensity[is_land],
    cmap="YlOrRd",
    s=10,
    alpha=0.8,
)

# 観測地点（遺跡）
ax.scatter(
    dataset.site_coords[:, 0],
    dataset.site_coords[:, 1],
    c="blue",
    s=30,
    marker="^",
    edgecolors="white",
    linewidths=0.5,
    alpha=0.8,
    label=f"遺跡 (n={dataset.num_sites()})",
)

# 境界
ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)

plt.colorbar(sc, ax=ax, label="強度 λ(s)")
ax.set_title(f"遺跡存在強度（{time_periods[period]}）\nλ* = {results.lambda_star_samples.mean():.2f}")
ax.set_xlabel("経度")
ax.set_ylabel("緯度")
ax.legend(loc="upper right")

for spine in ax.spines.values():
    spine.set_linewidth(0.5)

plt.show()

## 16. 全産地の一覧表示

In [ ]:
def plot_all_origins_grid():
    """全産地（4つ）を2×2でグリッド予測プロット"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12), constrained_layout=True)
    axes = axes.flatten()

    for idx in range(4):  # 4つの主要産地
        ax = axes[idx]
        
        # グリッド上の事後平均
        sc = ax.scatter(
            dataset.grid_coords[is_land, 0],
            dataset.grid_coords[is_land, 1],
            c=grid_probs[idx, is_land],
            cmap="Blues",
            s=8,
            alpha=0.8,
            vmin=0,
            vmax=1,
        )
        
        # 観測地点の実測比率
        ax.scatter(
            dataset.site_coords[:, 0],
            dataset.site_coords[:, 1],
            c=true_ratio[idx],
            cmap="Blues",
            s=30,
            edgecolors="black",
            linewidths=0.3,
            alpha=0.9,
            vmin=0,
            vmax=1,
        )
        
        # 境界
        ax.scatter(boundary[:, 0], boundary[:, 1], c="grey", s=0.001)

        plt.colorbar(sc, ax=ax, label="確率")
        ax.set_title(f"{origins[idx]}")
        ax.set_xlabel("経度")
        ax.set_ylabel("緯度")

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    fig.suptitle(f"黒曜石産地組成比のグリッド予測（{time_periods[period]}）", fontsize=14)
    plt.show()

plot_all_origins_grid()

## まとめ

このノートブックでは、距離事前分布を組み込んだマーク付き点過程モデルを実装しました。

### 実装した機能
1. **点過程部分**: 遺跡の存在確率と強度関数
2. **マーク部分**: 距離事前分布を用いた産地構成比のモデリング
3. **効果分解**: 距離効果とデータ駆動効果の分離
4. **空間効果**: NNGPによる空間相関

### 確認したこと
1. 距離ベース事前確率の可視化
2. MCMCの収束確認（トレースプロット）
3. 効果の分解と比較
4. 推定精度の評価（相関係数、RMSE）
5. 遺跡存在強度のマップ

### ハイパーパラメータチューニングのポイント
- **lengthscale**: 小さいほど局所的、大きいほど広域的
- **variance**: 大きいほど効果が強い
- **tau**: 距離事前分布の温度パラメータ
- **source_weights**: 産地の重要度